# 🌟 Northstar Services — Support Assistant (v0)

**Session 2 Task:** Building a foundational customer support triage pipeline using direct Google Gemini API integration, Pydantic structured output validation, externalized YAML prompt configurations, and real-time token/cost accounting.

## 1. Environment & Client Configuration
Load API credentials from `.env` and initialize the Google GenAI SDK client with the target model (`gemini-3.1-flash-lite`).

In [1]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

In [2]:
from google import genai

# Initialize Google GenAI client
client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

## 2. Structured Output Schema (Pydantic)
Define strict validation contracts for incoming customer inquiries. The model classifies each ticket by category, urgency level, sentiment, escalation need (`need_human`), and generates an executive summary.

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


class Triage(BaseModel):
    """Schema enforcing structured classification for incoming customer support messages."""
    category: Literal["billing", "technical", "account", "general"] = Field(
        description="The main topic of the customer's message."
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How quickly this ticket requires a response."
    )
    sentiment: Literal["negative", "neutral", "positive"] = Field(
        description="The customer's mood and emotional tone in the message."
    )
    need_human: bool = Field(
        description="True if this ticket must escalate to a human agent — e.g. refunds, billing disputes, cancellations, legal issues, or angry users."
    )
    summary: str = Field(
        description="A concise, one-line summary of what the customer is requesting."
    )

## 3. Externalized Prompt Configuration (YAML)
Separate prompt engineering instructions from Python application code by loading system instructions from `prompt.yml`.

In [5]:
import yaml
from pathlib import Path

# Resolve prompt.yml path safely whether executed from root or task directory
prompt_file = Path("prompt.yml") if Path("prompt.yml").exists() else Path("Session_2/Task/prompt.yml")
_PROMPTS = yaml.safe_load(prompt_file.read_text(encoding="utf-8"))

TRIAGE_SYSTEM = _PROMPTS["triage_system"]
REPLY_SYSTEM = _PROMPTS["reply_system"]

print("Prompts loaded successfully!")

Prompts loaded successfully!


## 4. Token & Cost Estimation Tracker
Track cumulative prompt (input) and candidate (output) tokens, and compute operational pricing based on Gemini pricing tiers ($0.25 / 1M input, $1.50 / 1M output).

In [6]:
IN_PRICE_PER_MTOK = 0.25     # $ per million input tokens
OUT_PRICE_PER_MTOK = 1.50    # $ per million output tokens
_totals = {"input": 0, "output": 0}

def track(usage) -> None:
    """Accumulate token usage counts from Gemini response metadata."""
    if usage:
        _totals["input"] += usage.prompt_token_count or 0
        _totals["output"] += usage.candidates_token_count or 0

def usage_report() -> str:
    """Format a human-readable summary of token consumption and estimated cost."""
    cost = (_totals["input"] / 1e6 * IN_PRICE_PER_MTOK
            + _totals["output"] / 1e6 * OUT_PRICE_PER_MTOK)
    inp = _totals["input"]
    out = _totals["output"]
    return f"{inp} in + {out} out tokens = ~${cost:.6f}"

## 5. Triage & Draft Reply Pipeline
- `triage()`: Submits the message with `response_schema=Triage` to guarantee valid JSON adhering to our Pydantic schema.
- `draft_reply()`: Generates a concise, empathetic customer response grounded in the triage context.

In [7]:
def triage(message: str) -> Triage:
    """Classify incoming customer message and return a validated Triage object."""
    resp = client.models.generate_content(
        model=MODEL,
        contents=message,
        config={
            "system_instruction": TRIAGE_SYSTEM,
            "response_mime_type": "application/json",
            "response_schema": Triage,
        },
    )
    track(resp.usage_metadata)
    return resp.parsed

In [8]:
def draft_reply(message: str, t: Triage) -> str:
    """Draft a 2-3 sentence empathetic initial response using the triage metadata."""
    resp = client.models.generate_content(
        model=MODEL,
        contents=f"Customer message:\n{message}\n\nTriage: {t.model_dump()}",
        config={"system_instruction": REPLY_SYSTEM},
    )
    track(resp.usage_metadata)
    return resp.text

## 6. End-to-End Evaluation
Evaluate the full triage and drafting pipeline across a representative test suite of 6 diverse customer inquiries.

In [11]:
def main() -> None:
    """Execute evaluation across sample support tickets and display token statistics."""
    samples = [
        "I've been charged twice for May and nobody has replied. This is ridiculous.",
        "How do I change the email address on my account?",
        "Your app keeps crashing when I upload a file.",
        "Who are you?",
        "What can you do for me",
        "Can I directly contact with the human support",
    ]
    for s in samples:
        t = triage(s)
        print("-" * 70)
        print("MESSAGE :", s)
        print("TRIAGE  :", t.model_dump())
        print("ESCALATE:", "yes -> human" if t.need_human else "no")
        print("DRAFT   :", draft_reply(s, t))

    print("=" * 70)
    print("TOKENS  :", usage_report())

main()

----------------------------------------------------------------------
MESSAGE : I've been charged twice for May and nobody has replied. This is ridiculous.
TRIAGE  : {'category': 'billing', 'urgency': 'high', 'sentiment': 'negative', 'need_human': True, 'summary': 'Customer reports a double charge for May and expresses frustration over lack of previous response.'}
ESCALATE: yes -> human
DRAFT   : Hello, I am very sorry for the frustration caused by this billing error and the delay in our response. A specialized teammate is currently reviewing your account details to resolve this matter for you, and they will follow up shortly with the next steps.
----------------------------------------------------------------------
MESSAGE : How do I change the email address on my account?
TRIAGE  : {'category': 'account', 'urgency': 'low', 'sentiment': 'neutral', 'need_human': False, 'summary': 'Customer is asking for instructions on how to update their account email address.'}
ESCALATE: no
DRAFT   